# AI Melanoma Diagnosis Pipeline

## Testing TransUNet Model

Pipeline:
ISIC 2016 Dataset → Pre-processing → TransUNet Segmentation → Lesion ROI → EfficientNetB0 Classification → Grad-CAM

### Purpose
This notebook evaluates TransUNet for skin-lesion segmentation and uses the segmented lesion as input to an EfficientNetB0 classifier for benign vs melanoma classification.

For fair comparison with the other U-Net architectures, the same dataset, preprocessing, data split, loss functions and evaluation metrics are used.

### 1. Setup and Import Libraries

In [ ]:
import os
import zipfile
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

# a fixed random seed is used to improve reproducibility between U-Net model comparisons.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# settings the same as the U-Net++ comparison
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 60

DATA_DIR = "/content/isic2016"
os.makedirs(DATA_DIR, exist_ok=True)

### 2. Connect to ISIC 2016 Dataset

In [ ]:
# Create a folder in Colab to store the ISIC 2016 dataset
DATA_DIR = "/content/isic2016"
os.makedirs(DATA_DIR, exist_ok=True)

# URLs for the ISIC 2016 images, expert segmentation masks,
# and benign/melanoma diagnosis labels
FILES = {
    "seg_train_images": (
        "ISBI2016_ISIC_Part1_Training_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_Data.zip"
    ),
    "seg_train_masks": (
        "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip"
    ),
    "seg_test_images": (
        "ISBI2016_ISIC_Part1_Test_Data.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_Data.zip"
    ),
    "seg_test_masks": (
        "ISBI2016_ISIC_Part1_Test_GroundTruth.zip",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_GroundTruth.zip"
    ),
    "cls_train_labels": (
        "ISBI2016_ISIC_Part3_Training_GroundTruth.csv",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Training_GroundTruth.csv"
    ),
    "cls_test_labels": (
        "ISBI2016_ISIC_Part3_Test_GroundTruth.csv",
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Test_GroundTruth.csv"
    )
}

print("Dataset folder created and ISIC 2016 file locations defined.")

In [ ]:
# Download each ISIC 2016 file into the Colab dataset folder
def download_file(url, destination):
    if os.path.exists(destination):
        print("Already downloaded:", os.path.basename(destination))
        return

    print("Downloading:", os.path.basename(destination))

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(destination, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)

    print("Finished:", os.path.basename(destination))


# Download all files listed above
for name, (filename, url) in FILES.items():
    destination = os.path.join(DATA_DIR, filename)
    download_file(url, destination)

### 3. Extract Dataset Files

In [ ]:
# Extract the downloaded image and mask ZIP files
zip_files = [
    "ISBI2016_ISIC_Part1_Training_Data.zip",
    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
    "ISBI2016_ISIC_Part1_Test_Data.zip",
    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip"
]

for filename in zip_files:
    zip_path = os.path.join(DATA_DIR, filename)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    print("Extracted:", filename)

### 4. Load and Inspect Data

In [ ]:
# Load the benign/melanoma diagnosis labels from ISIC Task 3
def load_labels(csv_path):
    df_labels = pd.read_csv(
        csv_path,
        header=None,
        names=["image_id", "label"]
    )

    # Convert diagnosis to binary:
    # 0 = benign, 1 = melanoma
    def convert_label(value):
        if isinstance(value, str):
            return 1 if value.strip().lower() == "malignant" else 0
        return int(float(value))

    df_labels["target"] = df_labels["label"].apply(convert_label)

    return df_labels.set_index("image_id")["target"].to_dict()


# Combine training and test diagnosis labels
label_lookup = {}

label_lookup.update(
    load_labels(os.path.join(
        DATA_DIR,
        "ISBI2016_ISIC_Part3_Training_GroundTruth.csv"
    ))
)

label_lookup.update(
    load_labels(os.path.join(
        DATA_DIR,
        "ISBI2016_ISIC_Part3_Test_GroundTruth.csv"
    ))
)

print("Diagnosis labels loaded:", len(label_lookup))

In [ ]:
# Find each dermoscopic image and match it with its expert mask and diagnosis
def find_image_mask_pairs(image_folder, mask_folder):
    image_files = sorted([
        f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    records = []

    for filename in image_files:
        image_id = os.path.splitext(filename)[0]

        # ISIC masks use the same image ID followed by "_Segmentation.png"
        mask_filename = image_id + "_Segmentation.png"
        mask_path = os.path.join(mask_folder, mask_filename)

        # Only keep cases where image, mask and diagnosis are all available
        if os.path.exists(mask_path) and image_id in label_lookup:
            records.append({
                "image_id": image_id,
                "filepath": os.path.join(image_folder, filename),
                "maskpath": mask_path,
                "target": label_lookup[image_id]
            })

    return records

In [ ]:
# Define the folders containing the training and test images/masks
train_image_dir = os.path.join(
    DATA_DIR, "ISBI2016_ISIC_Part1_Training_Data"
)

train_mask_dir = os.path.join(
    DATA_DIR, "ISBI2016_ISIC_Part1_Training_GroundTruth"
)

test_image_dir = os.path.join(
    DATA_DIR, "ISBI2016_ISIC_Part1_Test_Data"
)

test_mask_dir = os.path.join(
    DATA_DIR, "ISBI2016_ISIC_Part1_Test_GroundTruth"
)

# Match images, masks and diagnosis labels
records = (
    find_image_mask_pairs(train_image_dir, train_mask_dir)
    + find_image_mask_pairs(test_image_dir, test_mask_dir)
)

# Store all matched cases in a table
df = pd.DataFrame(records)

print("Total matched cases:", len(df))
print(df.head())

### 5. Visual Checks

In [ ]:
# Preview one image and mask pair to check they match correctly

sample = df.iloc[0]
image_id = sample["image_id"]

image = cv2.imread(sample["filepath"])
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mask = cv2.imread(sample["maskpath"], cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(f"Image {image_id}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.show()

In [ ]:
# Count diagnosis labels
# 0 = benign, 1 = melanoma

class_counts = df["target"].value_counts().sort_index()

benign_count = class_counts.get(0, 0)
melanoma_count = class_counts.get(1, 0)

print("Benign (0):", benign_count)
print("Melanoma (1):", melanoma_count)

# Plot dataset class distribution
plt.figure(figsize=(6, 4))

bars = plt.bar(
    ["Benign", "Melanoma"],
    [benign_count, melanoma_count],
    color=["blue", "red"]
)

# Add the number of cases above each bar
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height())}",
        ha="center",
        va="bottom"
    )

plt.ylabel("Number of Images")
plt.title("Class Distribution")
plt.show()

### 6. Load Images into NumPy Arrays

In [ ]:
# Set all images and masks to the same size for TransUNet
IMG_SIZE = 256

images = []
masks = []
labels = []

for _, row in df.iterrows():

    # Load and resize image to 256 × 256
    image = cv2.imread(row["filepath"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    # Normalise image pixel values from 0–255 to 0–1
    image = image.astype("float32") / 255.0

    # Load and resize corresponding mask
    mask = cv2.imread(row["maskpath"], cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    # Convert mask to binary: 0 = background, 1 = lesion
    mask = (mask > 127).astype("float32")

    images.append(image)
    masks.append(mask)
    labels.append(row["target"])

# Convert lists into NumPy arrays
images = np.array(images, dtype="float32")
masks = np.array(masks, dtype="float32")[..., np.newaxis]
labels = np.array(labels)

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Labels:", labels.shape)

### 7. Sample Visualisation

In [ ]:
# Find one benign and one melanoma example
benign_index = np.where(labels == 0)[0][0]
melanoma_index = np.where(labels == 1)[0][0]

sample_indices = [benign_index, melanoma_index]
sample_names = ["Benign", "Melanoma"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for row, (index, name) in enumerate(zip(sample_indices, sample_names)):

    image = images[index]
    mask = masks[index].squeeze()

    # Create red mask overlay
    overlay = image.copy()
    overlay[mask > 0.5] = (
        overlay[mask > 0.5] * 0.5
        + np.array([1, 0, 0]) * 0.5
    )

    # Original image
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"{name} - Image")
    axes[row, 0].axis("off")

    # Ground-truth lesion mask
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"{name} - Ground Truth Mask")
    axes[row, 1].axis("off")

    # Image with mask overlay
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"{name} - Image + Mask")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display a random benign and melanoma example
# A different random image is selected each time the cell is run

def plot_class_sample(images, masks, labels, class_value, class_name):
    idxs = np.where(labels == class_value)[0]

    # Randomly select an example from this class
    i = np.random.choice(idxs)

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(images[i])
    ax[0].set_title("Image")
    ax[0].axis("off")

    ax[1].imshow(masks[i, ..., 0], cmap="gray")
    ax[1].set_title("Ground-Truth Mask")
    ax[1].axis("off")

    ax[2].imshow(images[i])
    ax[2].imshow(masks[i, ..., 0], cmap="jet", alpha=0.4)
    ax[2].set_title("Image + Mask")
    ax[2].axis("off")

    fig.suptitle(f"{class_name} Example", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_class_sample(images, masks, labels, 0, "Benign")
plot_class_sample(images, masks, labels, 1, "Melanoma")

### 8. Train / Validation / Test Split (70 : 10 : 20)

In [ ]:
# Split dataset into 70% training, 10% validation and 20% testing
# Stratification keeps the benign/melanoma proportions consistent
# Random seed matches the U-Net++ model for a fair comparison

SEED = 42

X_temp, X_test, y_temp, y_test, t_temp, t_test = train_test_split(
    images,
    masks,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=SEED
)

X_train, X_val, y_train, y_val, t_train, t_val = train_test_split(
    X_temp,
    y_temp,
    t_temp,
    test_size=0.125,
    stratify=t_temp,
    random_state=SEED
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

### 9. Build TransUNet Model

In [ ]:
# ============================================================
# TRANSUNET MODEL
# CNN encoder + Transformer bottleneck + U-Net decoder
# ============================================================

# Model settings
IMG_SIZE = 256
PROJECTION_DIM = 256
NUM_HEADS = 8
TRANSFORMER_LAYERS = 4
MLP_DIM = 512


# ------------------------------------------------------------
# Convolution block
# Extracts local image features such as edges and textures
# ------------------------------------------------------------

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    return x


# ------------------------------------------------------------
# Transformer MLP block
# ------------------------------------------------------------

def mlp_block(x):
    x = layers.Dense(MLP_DIM, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.1)(x)

    x = layers.Dense(PROJECTION_DIM, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.1)(x)

    return x


# ------------------------------------------------------------
# Transformer encoder
# Learns global relationships between different lesion regions
# ------------------------------------------------------------

def transformer_encoder(x):

    x1 = layers.LayerNormalization(epsilon=1e-6)(x)

    attention = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=PROJECTION_DIM // NUM_HEADS,
        dropout=0.1
    )(x1, x1)

    x2 = layers.Add()([x, attention])

    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    x3 = mlp_block(x3)

    return layers.Add()([x2, x3])


# ------------------------------------------------------------
# Decoder block
# Upsamples features and combines them with CNN skip connections
# ------------------------------------------------------------

def decoder_block(x, skip, filters):

    x = layers.Conv2DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same"
    )(x)

    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)

    return x


# ------------------------------------------------------------
# Build TransUNet
# ------------------------------------------------------------

def build_transunet(input_shape=(256, 256, 3)):

    inputs = layers.Input(input_shape)

    # ----- CNN Encoder -----

    e1 = conv_block(inputs, 32)        # 256 × 256
    p1 = layers.MaxPooling2D(2)(e1)

    e2 = conv_block(p1, 64)            # 128 × 128
    p2 = layers.MaxPooling2D(2)(e2)

    e3 = conv_block(p2, 128)           # 64 × 64
    p3 = layers.MaxPooling2D(2)(e3)

    e4 = conv_block(p3, 256)           # 32 × 32
    p4 = layers.MaxPooling2D(2)(e4)    # 16 × 16


    # ----- Transformer Bottleneck -----

    x = layers.Conv2D(
        PROJECTION_DIM,
        kernel_size=1,
        padding="same"
    )(p4)

    # Convert 16 × 16 feature map into 256 Transformer tokens
    x = layers.Reshape((256, PROJECTION_DIM))(x)

    # Learnable positional embeddings
    positions = tf.range(start=0, limit=256, delta=1)

    position_embedding = layers.Embedding(
        input_dim=256,
        output_dim=PROJECTION_DIM
    )(positions)

    x = x + position_embedding

    # Transformer layers
    for _ in range(TRANSFORMER_LAYERS):
        x = transformer_encoder(x)

    # Convert tokens back into spatial feature map
    x = layers.Reshape((16, 16, PROJECTION_DIM))(x)


    # ----- U-Net Decoder -----

    d1 = decoder_block(x, e4, 256)     # 32 × 32
    d2 = decoder_block(d1, e3, 128)    # 64 × 64
    d3 = decoder_block(d2, e2, 64)     # 128 × 128
    d4 = decoder_block(d3, e1, 32)     # 256 × 256


    # ----- Segmentation Output -----

    outputs = layers.Conv2D(
        1,
        kernel_size=1,
        activation="sigmoid"
    )(d4)

    return models.Model(
        inputs,
        outputs,
        name="TransUNet"
    )


# Create model
transunet = build_transunet()

# Display architecture
transunet.summary()

### 10. (Dice / Jaccard Metrics)

In [ ]:
# Dice coefficient
# Measures overlap between the predicted and ground-truth lesion masks

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )


# Jaccard coefficient (Intersection over Union)
# Measures the intersection relative to the combined mask area

def jaccard_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    return (intersection + smooth) / (union + smooth)


# Combined Binary Cross-Entropy + Dice loss
# Encourages accurate pixel classification and good lesion-mask overlap

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coefficient(y_true, y_pred)

    return tf.reduce_mean(bce) + dice_loss

In [ ]:
# Compile TransUNet
# Adam optimizer with a learning rate of 1e-4

transunet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[
        dice_coefficient,
        jaccard_coefficient
    ]
)

print("TransUNet compiled successfully.")

In [ ]:
# Training callbacks
# Save the model with the best validation Dice score
# Stop training if validation performance stops improving
# Reduce the learning rate if improvement plateaus

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_transunet.keras",
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",
        mode="max",
        factor=0.5,
        patience=6,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# Train TransUNet

EPOCHS = 60
BATCH_SIZE = 16

history = transunet.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

### 11. Training Curves (Training vs Validation)

In [ ]:
# Plot training and validation Loss, Dice and Jaccard

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history["loss"], label="Training")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Dice coefficient
axes[1].plot(history.history["dice_coefficient"], label="Training")
axes[1].plot(history.history["val_dice_coefficient"], label="Validation")
axes[1].set_title("Dice Coefficient")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()

# Jaccard coefficient
axes[2].plot(history.history["jaccard_coefficient"], label="Training")
axes[2].plot(history.history["val_jaccard_coefficient"], label="Validation")
axes[2].set_title("Jaccard Coefficient")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Jaccard")
axes[2].legend()

plt.tight_layout()
plt.show()

### 12. Test-Set Evaluation

In [ ]:
# Evaluate the best TransUNet model on the unseen test set

test_results = transunet.evaluate(
    X_test,
    y_test,
    batch_size=16,
    verbose=1
)

print(f"Test Loss: {test_results[0]:.4f}")
print(f"Test Dice Coefficient: {test_results[1]:.4f}")
print(f"Test Jaccard Coefficient: {test_results[2]:.4f}")

### 13. Test-Set Sample Predictions

In [ ]:
# Generate predicted lesion masks for the test set

y_pred = transunet.predict(
    X_test,
    batch_size=16,
    verbose=1
)

# Convert probability maps into binary masks
y_pred_binary = (y_pred > 0.5).astype("float32")

print("Predicted masks:", y_pred_binary.shape)

In [ ]:
# Display random TransUNet segmentation results from the test set

sample_indices = np.random.choice(len(X_test), 3, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, i in enumerate(sample_indices):

    # Original image
    axes[row, 0].imshow(X_test[i])
    axes[row, 0].set_title("Image")
    axes[row, 0].axis("off")

    # Ground-truth mask
    axes[row, 1].imshow(y_test[i, ..., 0], cmap="gray")
    axes[row, 1].set_title("Ground-Truth Mask")
    axes[row, 1].axis("off")

    # TransUNet predicted mask
    axes[row, 2].imshow(y_pred_binary[i, ..., 0], cmap="gray")
    axes[row, 2].set_title("Predicted Mask")
    axes[row, 2].axis("off")

    # Prediction overlaid on original image
    axes[row, 3].imshow(X_test[i])
    axes[row, 3].imshow(
        y_pred_binary[i, ..., 0],
        cmap="jet",
        alpha=0.4
    )
    axes[row, 3].set_title("Prediction Overlay")
    axes[row, 3].axis("off")

plt.tight_layout()
plt.show()

### 14. Best, Median and Worst Segmentation Results

In [ ]:
# Calculate Dice score for each individual test image

def individual_dice(y_true, y_pred, smooth=1e-6):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    intersection = np.sum(y_true * y_pred)

    return (2.0 * intersection + smooth) / (
        np.sum(y_true) + np.sum(y_pred) + smooth
    )


dice_scores = np.array([
    individual_dice(y_test[i], y_pred_binary[i])
    for i in range(len(y_test))
])

print(f"Best Dice:   {dice_scores.max():.4f}")
print(f"Median Dice: {np.median(dice_scores):.4f}")
print(f"Worst Dice:  {dice_scores.min():.4f}")

In [ ]:
# Identify the best, median and worst segmentation examples

best_idx = np.argmax(dice_scores)
worst_idx = np.argmin(dice_scores)

median_value = np.median(dice_scores)
median_idx = np.argmin(np.abs(dice_scores - median_value))

selected = [
    (best_idx, "Best"),
    (median_idx, "Median"),
    (worst_idx, "Worst")
]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, (i, name) in enumerate(selected):

    # Original image
    axes[row, 0].imshow(X_test[i])
    axes[row, 0].set_title(f"{name} - Image")
    axes[row, 0].axis("off")

    # Ground-truth mask
    axes[row, 1].imshow(y_test[i, ..., 0], cmap="gray")
    axes[row, 1].set_title("Ground-Truth Mask")
    axes[row, 1].axis("off")

    # Predicted mask
    axes[row, 2].imshow(y_pred_binary[i, ..., 0], cmap="gray")
    axes[row, 2].set_title("Predicted Mask")
    axes[row, 2].axis("off")

    # Prediction overlay
    axes[row, 3].imshow(X_test[i])
    axes[row, 3].imshow(
        y_pred_binary[i, ..., 0],
        cmap="jet",
        alpha=0.4
    )
    axes[row, 3].set_title(f"Dice = {dice_scores[i]:.4f}")
    axes[row, 3].axis("off")

plt.tight_layout()
plt.show()

### Save TransUNet Model

In [ ]:
# Connect Google Drive to Colab

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Save the trained TransUNet model to the 43022 folder

MODEL_PATH = "/content/drive/MyDrive/43022/TransUNet_ISIC2016.keras"

transunet.save(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

### Rerun when restarting

In [ ]:
import tensorflow as tf

# Custom segmentation metrics and loss used when TransUNet was trained

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)

    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def jaccard_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    return (intersection + smooth) / (union + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coefficient(y_true, y_pred)

    return tf.reduce_mean(bce) + dice_loss

print("Custom functions loaded.")

In [ ]:
# Load the trained TransUNet model from Google Drive

MODEL_PATH = "/content/drive/MyDrive/43022/TransUNet_ISIC2016.keras"

transunet = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "dice_coefficient": dice_coefficient,
        "jaccard_coefficient": jaccard_coefficient,
        "bce_dice_loss": bce_dice_loss
    }
)

print("TransUNet loaded successfully.")

In [ ]:
# Check whether the ISIC 2016 dataset is already available

import os

DATA_DIR = "/content/isic2016"

print("Dataset exists:", os.path.exists(DATA_DIR))

if os.path.exists(DATA_DIR):
    print(os.listdir(DATA_DIR))

In [ ]:
# Download the ISIC 2016 dataset again for this Colab runtime

import os
import urllib.request

DATA_DIR = "/content/isic2016"
os.makedirs(DATA_DIR, exist_ok=True)

files = {
    "ISBI2016_ISIC_Part1_Training_Data.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_Data.zip",

    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip",

    "ISBI2016_ISIC_Part1_Test_Data.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_Data.zip",

    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_GroundTruth.zip",

    "ISBI2016_ISIC_Part3_Training_GroundTruth.csv":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Training_GroundTruth.csv",

    "ISBI2016_ISIC_Part3_Test_GroundTruth.csv":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Test_GroundTruth.csv"
}

for filename, url in files.items():
    path = os.path.join(DATA_DIR, filename)
    print("Downloading:", filename)
    urllib.request.urlretrieve(url, path)

print("Dataset downloaded successfully.")

In [ ]:
# Extract the ISIC 2016 image and segmentation-mask files

import zipfile

zip_files = [
    "ISBI2016_ISIC_Part1_Training_Data.zip",
    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
    "ISBI2016_ISIC_Part1_Test_Data.zip",
    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip"
]

for filename in zip_files:
    path = os.path.join(DATA_DIR, filename)

    with zipfile.ZipFile(path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    print("Extracted:", filename)

print("Dataset extracted successfully.")

In [ ]:
# Convert diagnosis labels to 0 = benign and 1 = melanoma/malignant

records = []

for _, row in labels_df.iterrows():
    image_id = str(row[id_col])
    diagnosis = str(row[label_col]).lower().strip()

    if image_id in image_map and image_id in mask_map:
        records.append({
            "image_id": image_id,
            "image_path": image_map[image_id],
            "mask_path": mask_map[image_id],
            "label": 0 if diagnosis == "benign" else 1
        })

df = pd.DataFrame(records)

print("Matched cases:", len(df))
print(df["label"].value_counts().sort_index())

In [ ]:
# Check exactly what was extracted from the ISIC downloads

for item in os.listdir(DATA_DIR):
    print(item)

In [ ]:
# Check the format of the training and test diagnosis CSVs

print("TRAINING CSV:")
print(train_labels.head())
print("\nColumns:", train_labels.columns.tolist())

print("\nTEST CSV:")
print(test_labels.head())
print("\nColumns:", test_labels.columns.tolist())

In [ ]:
# Reload both diagnosis CSVs correctly without treating the first case as a header

train_labels = pd.read_csv(train_csv, header=None, names=["image_id", "diagnosis"])
test_labels = pd.read_csv(test_csv, header=None, names=["image_id", "diagnosis"])

# Training CSV: benign = 0, malignant = 1
train_labels["label"] = (
    train_labels["diagnosis"]
    .str.lower()
    .str.strip()
    .map({"benign": 0, "malignant": 1})
)

# Test CSV already uses 0 = benign and 1 = malignant
test_labels["label"] = test_labels["diagnosis"].astype(float).astype(int)

labels_df = pd.concat([
    train_labels[["image_id", "label"]],
    test_labels[["image_id", "label"]]
], ignore_index=True)

print("Diagnosis records:", len(labels_df))
print(labels_df["label"].value_counts().sort_index())

In [ ]:
# Match all 1,279 diagnosis records with their image and segmentation mask

records = []

for _, row in labels_df.iterrows():
    image_id = str(row["image_id"])

    if image_id in image_map and image_id in mask_map:
        records.append({
            "image_id": image_id,
            "image_path": image_map[image_id],
            "mask_path": mask_map[image_id],
            "label": int(row["label"])
        })

df = pd.DataFrame(records)

print("Matched cases:", len(df))
print(df["label"].value_counts().sort_index())

In [ ]:
# Reload all images, masks and classification labels

import cv2
import numpy as np

IMG_SIZE = 256

X = []
y = []
targets = []

for _, row in df.iterrows():

    # Load and preprocess image
    image = cv2.imread(row["image_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype("float32") / 255.0

    # Load and preprocess segmentation mask
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )
    mask = (mask > 127).astype("float32")
    mask = np.expand_dims(mask, axis=-1)

    X.append(image)
    y.append(mask)
    targets.append(row["label"])

X = np.array(X, dtype="float32")
y = np.array(y, dtype="float32")
targets = np.array(targets, dtype="int32")

print("Images:", X.shape)
print("Masks:", y.shape)
print("Labels:", targets.shape)

In [ ]:
# Recreate the same 70:10:20 train, validation and test split

from sklearn.model_selection import train_test_split

SEED = 42

X_temp, X_test, y_temp, y_test, t_temp, t_test = train_test_split(
    X, y, targets,
    test_size=0.20,
    stratify=targets,
    random_state=SEED
)

X_train, X_val, y_train, y_val, t_train, t_val = train_test_split(
    X_temp, y_temp, t_temp,
    test_size=0.125,
    stratify=t_temp,
    random_state=SEED
)

print("Training:", X_train.shape, t_train.shape)
print("Validation:", X_val.shape, t_val.shape)
print("Testing:", X_test.shape, t_test.shape)

In [ ]:
# Generate TransUNet lesion masks for the classifier dataset

train_masks_pred = transunet.predict(X_train, batch_size=16, verbose=1)
val_masks_pred = transunet.predict(X_val, batch_size=16, verbose=1)
test_masks_pred = transunet.predict(X_test, batch_size=16, verbose=1)

# Convert probabilities to binary lesion masks
train_masks_binary = (train_masks_pred > 0.5).astype("float32")
val_masks_binary = (val_masks_pred > 0.5).astype("float32")
test_masks_binary = (test_masks_pred > 0.5).astype("float32")

print("Training masks:", train_masks_binary.shape)
print("Validation masks:", val_masks_binary.shape)
print("Testing masks:", test_masks_binary.shape)

In [ ]:
# Apply TransUNet masks to retain the lesion region for classification

X_train_roi = X_train * train_masks_binary
X_val_roi = X_val * val_masks_binary
X_test_roi = X_test * test_masks_binary

print("Training ROI:", X_train_roi.shape)
print("Validation ROI:", X_val_roi.shape)
print("Testing ROI:", X_test_roi.shape)

In [ ]:
# Resize lesion ROI images from 256×256 to 224×224 for EfficientNetB0

CLASSIFIER_SIZE = 224

def resize_for_classifier(images):
    return np.array([
        cv2.resize(img, (CLASSIFIER_SIZE, CLASSIFIER_SIZE))
        for img in images
    ], dtype="float32")

X_train_cls = resize_for_classifier(X_train_roi)
X_val_cls = resize_for_classifier(X_val_roi)
X_test_cls = resize_for_classifier(X_test_roi)

print("Training:", X_train_cls.shape, t_train.shape)
print("Validation:", X_val_cls.shape, t_val.shape)
print("Testing:", X_test_cls.shape, t_test.shape)

In [ ]:
# Build EfficientNetB0 classifier for benign vs melanoma classification

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained EfficientNetB0 initially
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

classifier = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0_Classifier"
)

print("EfficientNetB0 classifier created.")
print("Total parameters:", classifier.count_params())

In [ ]:
# Compile EfficientNetB0 classifier

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="sensitivity"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("EfficientNetB0 classifier compiled successfully.")

In [ ]:
# Calculate class weights to account for benign/melanoma class imbalance

from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(t_train),
    y=t_train
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("Benign weight:", round(class_weights[0], 3))
print("Melanoma weight:", round(class_weights[1], 3))

In [ ]:
# Train the EfficientNetB0 classifier

classifier_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnetb0.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

history_cls = classifier.fit(
    X_train_cls,
    t_train,
    validation_data=(X_val_cls, t_val),
    epochs=30,
    batch_size=16,
    class_weight=class_weights,
    callbacks=classifier_callbacks,
    verbose=1
)

In [ ]:
# Generate melanoma probabilities for the held-out test set

y_prob = classifier.predict(
    X_test_cls,
    batch_size=16,
    verbose=1
).ravel()

# Convert probabilities to benign (0) or melanoma (1)
y_pred_cls = (y_prob >= 0.5).astype(int)

print("Predictions generated:", len(y_pred_cls))
print("Predicted benign:", np.sum(y_pred_cls == 0))
print("Predicted melanoma:", np.sum(y_pred_cls == 1))

In [ ]:
# Correct input scaling for Keras EfficientNetB0
# Our ROI images are currently 0–1; EfficientNetB0 expects 0–255 inputs

X_train_cls = X_train_cls * 255.0
X_val_cls = X_val_cls * 255.0
X_test_cls = X_test_cls * 255.0

print("Training range:", X_train_cls.min(), "to", X_train_cls.max())
print("Validation range:", X_val_cls.min(), "to", X_val_cls.max())
print("Testing range:", X_test_cls.min(), "to", X_test_cls.max())

In [ ]:
# Rebuild a fresh EfficientNetB0 classifier using correctly scaled inputs

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Keep pretrained feature extractor frozen initially
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

classifier = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0_Classifier"
)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="sensitivity"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Fresh EfficientNetB0 classifier ready.")
print("Parameters:", classifier.count_params())

In [ ]:
# Train EfficientNetB0 using correctly scaled ROI images

classifier_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnetb0_corrected.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

history_cls = classifier.fit(
    X_train_cls,
    t_train,
    validation_data=(X_val_cls, t_val),
    epochs=30,
    batch_size=16,
    class_weight=class_weights,
    callbacks=classifier_callbacks,
    verbose=1
)

In [ ]:
# Evaluate EfficientNetB0 on the held-out test set

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    recall_score,
    roc_auc_score
)

# Predict probability of melanoma
y_prob = classifier.predict(
    X_test_cls,
    batch_size=16,
    verbose=1
).ravel()

# Classification threshold: 0.5
y_pred_cls = (y_prob >= 0.5).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(t_test, y_pred_cls).ravel()

accuracy = accuracy_score(t_test, y_pred_cls)
sensitivity = recall_score(t_test, y_pred_cls)
specificity = tn / (tn + fp)
auc = roc_auc_score(t_test, y_prob)

print(f"Test Accuracy:    {accuracy:.4f}")
print(f"Test Sensitivity: {sensitivity:.4f}")
print(f"Test Specificity: {specificity:.4f}")
print(f"Test AUC:         {auc:.4f}")

print("\nConfusion Matrix:")
print("TN:", tn, " FP:", fp)
print("FN:", fn, " TP:", tp)

In [ ]:
# Plot classifier confusion matrix

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(t_test, y_pred_cls)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign", "Melanoma"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("EfficientNetB0 Classification Confusion Matrix")
plt.show()

In [ ]:
# Find the final 4D feature-map layer for Grad-CAM

for layer in reversed(base_model.layers):
    if len(layer.output.shape) == 4:
        print("Grad-CAM layer:", layer.name)
        print("Output shape:", layer.output.shape)
        break

In [ ]:
# Create Grad-CAM heatmaps from EfficientNetB0's final convolutional feature maps

GRADCAM_LAYER = "top_activation"

def make_gradcam_heatmap(img_array, model, base_model, layer_name):

    # Model that returns both the final convolutional features
    # and the classifier prediction
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            base_model.get_layer(layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)

        # Binary classifier output = probability of melanoma
        class_output = predictions[:, 0]

    # Calculate gradients of melanoma prediction
    # with respect to the convolutional feature maps
    grads = tape.gradient(class_output, conv_outputs)

    # Average gradient importance for each feature channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    # Weight feature maps by their importance
    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    # Keep positive contributions and normalise to 0–1
    heatmap = tf.maximum(heatmap, 0)

    if tf.reduce_max(heatmap) > 0:
        heatmap = heatmap / tf.reduce_max(heatmap)

    return heatmap.numpy()

print("Grad-CAM function ready.")

In [ ]:
# Select one correctly classified melanoma example

melanoma_correct = np.where((t_test == 1) & (y_pred_cls == 1))[0]

idx = melanoma_correct[0]

# Prepare image for Grad-CAM
img_input = np.expand_dims(X_test_cls[idx], axis=0)

heatmap = make_gradcam_heatmap(
    img_input,
    classifier,
    base_model,
    GRADCAM_LAYER
)

print("Test index:", idx)
print("True label: Melanoma")
print(f"Predicted melanoma probability: {y_prob[idx]:.3f}")
print("Heatmap shape:", heatmap.shape)

In [ ]:
# Overlay the Grad-CAM heatmap onto the melanoma image

import matplotlib.pyplot as plt
import cv2

# Resize heatmap to image dimensions
heatmap_resized = cv2.resize(heatmap, (224, 224))

# Display image + heatmap overlay
plt.figure(figsize=(6, 6))

plt.imshow(X_test_cls[idx] / 255.0)
plt.imshow(
    heatmap_resized,
    cmap="jet",
    alpha=0.45
)

plt.title(
    f"Grad-CAM – Melanoma\n"
    f"Predicted Melanoma Probability: {y_prob[idx]:.3f}"
)

plt.axis("off")
plt.show()

In [ ]:
# Find examples for a balanced Grad-CAM comparison figure

correct_benign = np.where((t_test == 0) & (y_pred_cls == 0))[0]
correct_melanoma = np.where((t_test == 1) & (y_pred_cls == 1))[0]
false_positive = np.where((t_test == 0) & (y_pred_cls == 1))[0]
false_negative = np.where((t_test == 1) & (y_pred_cls == 0))[0]

print("Correct benign:", len(correct_benign))
print("Correct melanoma:", len(correct_melanoma))
print("False positives:", len(false_positive))
print("False negatives:", len(false_negative))

In [ ]:
# Create Grad-CAM comparison showing correct and incorrect classifications

examples = [
    (correct_benign[0], "Correct Benign"),
    (correct_melanoma[0], "Correct Melanoma"),
    (false_positive[0], "False Positive"),
    (false_negative[0], "False Negative")
]

plt.figure(figsize=(16, 4))

for i, (example_idx, title) in enumerate(examples):

    # Generate Grad-CAM
    img_input = np.expand_dims(X_test_cls[example_idx], axis=0)

    example_heatmap = make_gradcam_heatmap(
        img_input,
        classifier,
        base_model,
        GRADCAM_LAYER
    )

    example_heatmap = cv2.resize(example_heatmap, (224, 224))

    # Display ROI with Grad-CAM overlay
    plt.subplot(1, 4, i + 1)

    plt.imshow(X_test_cls[example_idx] / 255.0)
    plt.imshow(example_heatmap, cmap="jet", alpha=0.45)

    true_class = "Melanoma" if t_test[example_idx] == 1 else "Benign"

    plt.title(
        f"{title}\n"
        f"True: {true_class}\n"
        f"Melanoma probability: {y_prob[example_idx]:.3f}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

# Custom functions required to reload the saved TransUNet model

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)

    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def jaccard_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    return (intersection + smooth) / (union + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coefficient(y_true, y_pred)

    return tf.reduce_mean(bce) + dice_loss

print("Custom functions loaded.")


In [ ]:
# Load the previously trained TransUNet model from Google Drive

MODEL_PATH = "/content/drive/MyDrive/43022/TransUNet_ISIC2016.keras"

transunet = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "dice_coefficient": dice_coefficient,
        "jaccard_coefficient": jaccard_coefficient,
        "bce_dice_loss": bce_dice_loss
    }
)

print("TransUNet loaded successfully.")

In [ ]:
# Download the ISIC 2016 dataset for this Colab runtime

import os
import urllib.request

DATA_DIR = "/content/isic2016"
os.makedirs(DATA_DIR, exist_ok=True)

files = {
    "ISBI2016_ISIC_Part1_Training_Data.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_Data.zip",

    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip",

    "ISBI2016_ISIC_Part1_Test_Data.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_Data.zip",

    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part1_Test_GroundTruth.zip",

    "ISBI2016_ISIC_Part3_Training_GroundTruth.csv":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Training_GroundTruth.csv",

    "ISBI2016_ISIC_Part3_Test_GroundTruth.csv":
        "https://isic-challenge-data.s3.amazonaws.com/2016/ISBI2016_ISIC_Part3_Test_GroundTruth.csv"
}

for filename, url in files.items():
    path = os.path.join(DATA_DIR, filename)

    if not os.path.exists(path):
        print("Downloading:", filename)
        urllib.request.urlretrieve(url, path)

print("Dataset downloaded successfully.")

In [ ]:
# Check which downloaded ZIP files are valid

import os
import zipfile

zip_files = [
    "ISBI2016_ISIC_Part1_Training_Data.zip",
    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
    "ISBI2016_ISIC_Part1_Test_Data.zip",
    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip"
]

for filename in zip_files:
    path = os.path.join(DATA_DIR, filename)

    print(
        filename,
        "| Size:", os.path.getsize(path), "bytes",
        "| Valid ZIP:", zipfile.is_zipfile(path)
    )

In [ ]:
# Extract the ISIC 2016 images and segmentation masks

import zipfile

zip_files = [
    "ISBI2016_ISIC_Part1_Training_Data.zip",
    "ISBI2016_ISIC_Part1_Training_GroundTruth.zip",
    "ISBI2016_ISIC_Part1_Test_Data.zip",
    "ISBI2016_ISIC_Part1_Test_GroundTruth.zip"
]

for filename in zip_files:
    path = os.path.join(DATA_DIR, filename)

    with zipfile.ZipFile(path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    print("Extracted:", filename)

print("Dataset extracted successfully.")

In [ ]:
# Extract all four valid ZIP files

for filename in zip_files:
    path = os.path.join(DATA_DIR, filename)

    with zipfile.ZipFile(path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    print("Extracted:", filename)

print("Dataset extracted successfully.")

In [ ]:
# Load and combine the benign/melanoma diagnosis labels

import pandas as pd

train_csv = os.path.join(
    DATA_DIR,
    "ISBI2016_ISIC_Part3_Training_GroundTruth.csv"
)

test_csv = os.path.join(
    DATA_DIR,
    "ISBI2016_ISIC_Part3_Test_GroundTruth.csv"
)

# Training CSV uses words: benign / malignant
train_labels = pd.read_csv(
    train_csv,
    header=None,
    names=["image_id", "diagnosis"]
)

train_labels["label"] = (
    train_labels["diagnosis"]
    .str.lower()
    .str.strip()
    .map({"benign": 0, "malignant": 1})
)

# Test CSV uses numeric labels: 0 / 1
test_labels = pd.read_csv(
    test_csv,
    header=None,
    names=["image_id", "diagnosis"]
)

test_labels["label"] = (
    test_labels["diagnosis"]
    .astype(float)
    .astype(int)
)

# Combine them
labels_df = pd.concat([
    train_labels[["image_id", "label"]],
    test_labels[["image_id", "label"]]
], ignore_index=True)

print("Total cases:", len(labels_df))
print("\nClass distribution:")
print(labels_df["label"].value_counts().sort_index())

In [ ]:
# Find all dermoscopic images and segmentation masks
# and match them to the diagnosis labels

import glob

image_files = glob.glob(
    os.path.join(DATA_DIR, "**", "*.jpg"),
    recursive=True
)

mask_files = glob.glob(
    os.path.join(DATA_DIR, "**", "*Segmentation.png"),
    recursive=True
)

# Create lookup dictionaries using ISIC image ID
image_map = {
    os.path.splitext(os.path.basename(path))[0]: path
    for path in image_files
}

mask_map = {
    os.path.basename(path).replace("_Segmentation.png", ""): path
    for path in mask_files
}

# Add paths to our label dataframe
labels_df["image_path"] = labels_df["image_id"].map(image_map)
labels_df["mask_path"] = labels_df["image_id"].map(mask_map)

# Keep cases where both image and mask were found
df = labels_df.dropna(
    subset=["image_path", "mask_path"]
).reset_index(drop=True)

print("Images found:", len(image_map))
print("Masks found:", len(mask_map))
print("Matched cases:", len(df))

In [ ]:
# Load and preprocess all images and segmentation masks
# Images: RGB, 256x256, normalised to 0–1
# Masks: grayscale, 256x256, binary 0/1

import cv2
import numpy as np

IMG_SIZE = 256

X = []
y = []
targets = []

for _, row in df.iterrows():

    # Load dermoscopic image
    image = cv2.imread(row["image_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype(np.float32) / 255.0

    # Load ground-truth segmentation mask
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    X.append(image)
    y.append(mask)
    targets.append(row["label"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
targets = np.array(targets, dtype=np.int32)

print("Images:", X.shape)
print("Masks:", y.shape)
print("Labels:", targets.shape)
print("Image range:", X.min(), "to", X.max())

In [ ]:
# Recreate the exact original train / validation / test split

from sklearn.model_selection import train_test_split

SEED = 42

# 80% development set, 20% held-out test set
X_temp, X_test, y_temp, y_test, t_temp, t_test = train_test_split(
    X,
    y,
    targets,
    test_size=0.20,
    stratify=targets,
    random_state=SEED
)

# Development set -> 70% training + 10% validation overall
X_train, X_val, y_train, y_val, t_train, t_val = train_test_split(
    X_temp,
    y_temp,
    t_temp,
    test_size=0.125,
    stratify=t_temp,
    random_state=SEED
)

print("Training:", X_train.shape, t_train.shape)
print("Validation:", X_val.shape, t_val.shape)
print("Testing:", X_test.shape, t_test.shape)

print("\nClass counts:")
print("Train - Benign:", np.sum(t_train == 0), "Melanoma:", np.sum(t_train == 1))
print("Val   - Benign:", np.sum(t_val == 0), "Melanoma:", np.sum(t_val == 1))
print("Test  - Benign:", np.sum(t_test == 0), "Melanoma:", np.sum(t_test == 1))

In [ ]:
# Generate lesion masks using the saved TransUNet
# No augmentation is applied here — these are our original predicted ROIs

print("Predicting training masks...")
train_masks_pred = transunet.predict(
    X_train, batch_size=16, verbose=1
)

print("Predicting validation masks...")
val_masks_pred = transunet.predict(
    X_val, batch_size=16, verbose=1
)

print("Predicting test masks...")
test_masks_pred = transunet.predict(
    X_test, batch_size=16, verbose=1
)

# Convert probabilities into binary lesion masks
train_masks_binary = (train_masks_pred >= 0.5).astype(np.float32)
val_masks_binary = (val_masks_pred >= 0.5).astype(np.float32)
test_masks_binary = (test_masks_pred >= 0.5).astype(np.float32)

print("\nPredicted mask shapes:")
print("Train:", train_masks_binary.shape)
print("Validation:", val_masks_binary.shape)
print("Test:", test_masks_binary.shape)

In [ ]:
# Apply the predicted TransUNet masks to isolate the lesion ROI

X_train_roi = X_train * train_masks_binary
X_val_roi = X_val * val_masks_binary
X_test_roi = X_test * test_masks_binary

print("ROI image shapes:")
print("Train:", X_train_roi.shape)
print("Validation:", X_val_roi.shape)
print("Test:", X_test_roi.shape)

In [ ]:
# Resize TransUNet ROI images for EfficientNetB0
# Convert from 0–1 back to 0–255 for Keras EfficientNetB0

CLS_SIZE = 224

X_train_cls = tf.image.resize(X_train_roi, (CLS_SIZE, CLS_SIZE)).numpy()
X_val_cls = tf.image.resize(X_val_roi, (CLS_SIZE, CLS_SIZE)).numpy()
X_test_cls = tf.image.resize(X_test_roi, (CLS_SIZE, CLS_SIZE)).numpy()

X_train_cls *= 255.0
X_val_cls *= 255.0
X_test_cls *= 255.0

print("Classifier input shapes:")
print("Train:", X_train_cls.shape)
print("Validation:", X_val_cls.shape)
print("Test:", X_test_cls.shape)

print("\nTraining image range:")
print(X_train_cls.min(), "to", X_train_cls.max())

In [ ]:
# Data augmentation for TRAINING images only
# These transformations introduce realistic variation without changing
# whether the lesion is benign or melanoma.

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    ),
    tf.keras.layers.RandomContrast(0.10)
], name="data_augmentation")

print("Data augmentation created successfully.")


In [ ]:
# Build a fresh EfficientNetB0 classifier WITH data augmentation

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model

# Pretrained ImageNet backbone
base_model_aug = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Keep backbone frozen for a fair comparison with our original classifier
base_model_aug.trainable = False

# Input
inputs = tf.keras.Input(shape=(224, 224, 3))

# NEW: augmentation is applied only while training
x = data_augmentation(inputs)

# EfficientNetB0 feature extraction
x = base_model_aug(x, training=False)

# Classification head — same as original experiment
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

classifier_aug = Model(
    inputs,
    outputs,
    name="EfficientNetB0_Augmented"
)

print("Augmented EfficientNetB0 created.")
print("Total parameters:", classifier_aug.count_params())

In [ ]:
# Compile augmented EfficientNetB0 using the same settings
# as the original no-augmentation classifier

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, AUC

classifier_aug.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        Recall(name="sensitivity"),
        AUC(name="auc")
    ]
)

print("Augmented classifier compiled successfully.")

In [ ]:
# Calculate class weights to account for the imbalance
# between benign and melanoma training cases

from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(t_train),
    y=t_train
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("Class weights:")
print("Benign (0):", round(class_weights[0], 3))
print("Melanoma (1):", round(class_weights[1], 3))

In [ ]:
# Train EfficientNetB0 WITH data augmentation

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_path_aug = "/content/drive/MyDrive/43022/EfficientNetB0_Augmented.keras"

callbacks_aug = [
    ModelCheckpoint(
        checkpoint_path_aug,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

history_aug = classifier_aug.fit(
    X_train_cls,
    t_train,

    validation_data=(X_val_cls, t_val),

    epochs=30,
    batch_size=16,

    class_weight=class_weights,

    callbacks=callbacks_aug,
    verbose=1
)

In [ ]:
# Evaluate the augmented EfficientNetB0 on the untouched test set
# Threshold = 0.5, identical to the original classifier evaluation

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    recall_score,
    roc_auc_score
)

# Predict melanoma probabilities
y_prob_aug = classifier_aug.predict(
    X_test_cls,
    batch_size=16,
    verbose=1
).ravel()

# Convert probabilities to classifications
y_pred_aug = (y_prob_aug >= 0.5).astype(int)

# Confusion matrix
tn_aug, fp_aug, fn_aug, tp_aug = confusion_matrix(
    t_test,
    y_pred_aug
).ravel()

# Calculate metrics
accuracy_aug = accuracy_score(t_test, y_pred_aug)
sensitivity_aug = recall_score(t_test, y_pred_aug)
specificity_aug = tn_aug / (tn_aug + fp_aug)
auc_aug = roc_auc_score(t_test, y_prob_aug)

print("\nAUGMENTED MODEL — TEST RESULTS")
print("--------------------------------")
print(f"Accuracy:    {accuracy_aug:.4f} ({accuracy_aug*100:.2f}%)")
print(f"Sensitivity: {sensitivity_aug:.4f} ({sensitivity_aug*100:.2f}%)")
print(f"Specificity: {specificity_aug:.4f} ({specificity_aug*100:.2f}%)")
print(f"AUC:         {auc_aug:.4f}")

print("\nConfusion Matrix:")
print("TN:", tn_aug)
print("FP:", fp_aug)
print("FN:", fn_aug)
print("TP:", tp_aug)

In [ ]:
# Confusion matrix for EfficientNetB0 WITH data augmentation

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm_aug = confusion_matrix(t_test, y_pred_aug)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_aug,
    display_labels=["Benign", "Melanoma"]
)

disp.plot(values_format="d")
plt.title("EfficientNetB0 with Data Augmentation")
plt.show()

In [ ]:
# Confusion matrix for EfficientNetB0 WITH data augmentation
# Same colour scheme/style as the original figure

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_aug = confusion_matrix(t_test, y_pred_aug)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_aug,
    display_labels=["Benign", "Melanoma"]
)

disp.plot(
    cmap="Blues",
    values_format="d",
    colorbar=True
)

plt.title("EfficientNetB0 Classification Confusion Matrix\nWith Data Augmentation")
plt.xlabel("Predicted label")
plt.ylabel("True label")

plt.show()

In [ ]:
# ROC curve for EfficientNetB0 WITH data augmentation

from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

fpr_aug, tpr_aug, _ = roc_curve(t_test, y_prob_aug)
auc_aug = roc_auc_score(t_test, y_prob_aug)

plt.figure(figsize=(6, 5))

plt.plot(
    fpr_aug,
    tpr_aug,
    label=f"EfficientNetB0 + Augmentation (AUC = {auc_aug:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – EfficientNetB0 With Data Augmentation")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Training curves for EfficientNetB0 WITH data augmentation

import matplotlib.pyplot as plt

epochs = range(1, len(history_aug.history["loss"]) + 1)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_aug.history["auc"],
    label="Training AUC"
)

plt.plot(
    epochs,
    history_aug.history["val_auc"],
    label="Validation AUC"
)

plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("EfficientNetB0 Training – With Data Augmentation")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Compare classifier performance before and after data augmentation

import matplotlib.pyplot as plt
import numpy as np

metrics = ["Accuracy", "Sensitivity", "Specificity", "AUC"]

no_aug = [
    0.7344,
    0.7000,
    0.7427,
    0.7755
]

with_aug = [
    accuracy_aug,
    sensitivity_aug,
    specificity_aug,
    auc_aug
]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(8, 5))

bars1 = plt.bar(
    x - width/2,
    no_aug,
    width,
    label="No Augmentation"
)

bars2 = plt.bar(
    x + width/2,
    with_aug,
    width,
    label="With Augmentation"
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(x, metrics)
plt.title("Effect of Data Augmentation on Classification Performance")
plt.legend()

# Add values above bars
for bars in [bars1, bars2]:
    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.015,
            f"{bar.get_height():.3f}",
            ha="center"
        )

plt.show()

In [ ]:
# Final results organised according to the project aims

import pandas as pd

results = pd.DataFrame({
    "Assessment Aim": [
        "Accurate ROI",
        "Accurate ROI",
        "Rate of Error",
        "Rate of Error",
        "Rate of Error",
        "Rate of Error",
        "Classification Accuracy",
        "Classification Accuracy"
    ],

    "Metric": [
        "Dice",
        "IoU",
        "False Positives",
        "False Negatives",
        "Sensitivity",
        "Specificity",
        "Accuracy",
        "AUC"
    ],

    "No Augmentation": [
        0.8850,
        0.7970,
        53,
        15,
        0.7000,
        0.7427,
        0.7344,
        0.7755
    ],

    "With Augmentation": [
        0.8850,   # Segmentation unchanged
        0.7970,   # Segmentation unchanged
        fp_aug,
        fn_aug,
        sensitivity_aug,
        specificity_aug,
        accuracy_aug,
        auc_aug
    ]
})

print(results.to_string(index=False))

In [ ]:
# Save the final EfficientNetB0 model WITH data augmentation
# This contains the best restored weights from epoch 9

FINAL_MODEL_PATH = "/content/drive/MyDrive/43022/EfficientNetB0_Augmented_Final.keras"

classifier_aug.save(FINAL_MODEL_PATH)

print("Model saved successfully.")
print("Location:", FINAL_MODEL_PATH)